# 02. 모델(LLM) — LangChain V1 모델 완전 가이드

> LangChain V1의 모델 계층은 `init_chat_model("provider:model")` 한 줄로 통일됐어요. invoke/stream/batch와 Content Blocks까지 모델을 다루는 패턴을 정리해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `init_chat_model`로 20가지 이상의 제공자를 동일한 방식으로 초기화할 수 있어요
2. `invoke`, `stream`, `ainvoke`, `astream`, `batch` 5가지 호출 방식을 상황에 맞게 선택할 수 있어요
3. `bind_tools`와 `with_structured_output`으로 Tool Calling과 구조화 출력을 구현할 수 있어요
4. `UsageMetadataCallbackHandler`로 토큰 사용량을 추적하고 비용을 관리할 수 있어요
5. Logprobs와 멀티모달 입력을 활용할 수 있어요
6. OpenAI 호환 API로 커스텀 엔드포인트와 사내 모델 서버를 연결할 수 있어요

## 사전 지식

- LangChain V0 기본 사용 경험
- 이전 노트북: `01-LangGraph-Python-Basics.ipynb` — TypedDict, Annotated, add_messages


## LLM과 LangChain V1 모델 아키텍처

LLM(Large Language Model)은 에이전트의 추론 엔진이에요. LangChain V1에서는 모든 제공자를 `init_chat_model` 하나로 통합했어요.

```mermaid
flowchart LR
    A["사용자 입력<br>User Input"] --> B["init_chat_model<br>'provider:model'"]:::process
    B --> C{"호출 방식<br>Call Mode"}:::process
    C --> D["invoke<br>동기 전체 응답"]:::output
    C --> E["stream<br>실시간 스트리밍"]:::output
    C --> F["ainvoke / astream<br>비동기 처리"]:::output
    C --> G["batch<br>병렬 일괄 처리"]:::output
    D --> H["AIMessage<br>응답 객체"]:::storage
    E --> H
    F --> H
    G --> H
    H --> I["Tool Calling<br>도구 호출"]:::process
    H --> J["Structured Output<br>구조화 출력"]:::process

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### 핵심 구성 요소

| 구성 요소 | 설명 | V1 변경 사항 |
|-----------|------|--------------|
| `init_chat_model` | 통합 모델 초기화 함수 | `ChatOpenAI`, `ChatAnthropic` 직접 임포트 불필요 |
| `provider:model` 형식 | `"openai:gpt-4o-mini"` 처럼 간결하게 지정 | 제공자 자동 감지 |
| `AIMessage` | 모든 응답의 표준 객체 | `content_blocks` 속성 추가 |
| `UsageMetadataCallbackHandler` | 토큰 사용량 누적 추적 | 모델별 분리 집계 지원 |

> 🔑 **핵심 개념**: LangChain V1에서는 `from langchain.chat_models import init_chat_model`로 모든 제공자를 통일된 방식으로 사용해요. 제공자별 패키지를 직접 import할 필요가 없어서 코드가 훨씬 간결해졌어요.


## 환경 설정


In [1]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# dotenv: .env 파일에서 API 키를 로드해요
from dotenv import load_dotenv
import os

# .env 파일의 환경 변수를 로드해요
load_dotenv(override=True)

# LangSmith 추적 설정 (선택 사항)
# LangSmith에서 모델 호출 과정을 시각적으로 디버깅할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Part02-Models"


True

## 1. 모델 초기화: init_chat_model

LangChain V1의 핵심 변화 중 하나는 `init_chat_model` 함수예요. `provider:model` 형식의 문자열 하나로 20가지 이상의 제공자를 동일하게 초기화할 수 있어요.

### 지원 제공자 목록

| 제공자 | 형식 | 패키지 |
|--------|------|---------|
| OpenAI | `openai:gpt-4o-mini` | `langchain-openai` |
| Anthropic | `anthropic:claude-sonnet-4-5` | `langchain-anthropic` |
| Google Gemini | `google_genai:gemini-2.0-flash` | `langchain-google-genai` |
| Groq | `groq:llama-3.3-70b-versatile` | `langchain-groq` |
| AWS Bedrock | `bedrock:us.amazon.nova-lite-v1:0` | `langchain-aws` |
| Mistral | `mistralai:mistral-large-latest` | `langchain-mistralai` |
| DeepSeek | `deepseek:deepseek-chat` | `langchain-deepseek` |


In [2]:
# ---------------------------------------------------
# 모델 초기화: init_chat_model
# ---------------------------------------------------
# V1 통합 임포트 경로 사용 (langchain_core가 아닌 langchain)
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델로 교체할 때는 문자열만 바꾸면 돼요!
# 예: "anthropic:claude-sonnet-4-5"
# 예: "google_genai:gemini-2.0-flash"
# 예: "groq:llama-3.3-70b-versatile"
model = init_chat_model("openai:gpt-4o-mini")

print(f"모델 타입: {type(model).__name__}")
print(f"모델 이름: {model.model_name}")


모델 타입: ChatOpenAI
모델 이름: gpt-4o-mini


In [3]:
from langchain.chat_models import init_chat_model

model_detailed = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.1,
    timeout=30,
    max_retries=2
)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 모델 세부 설정
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 기본 호출: invoke()

`invoke()`는 동기(synchronous) 방식으로 전체 응답이 생성될 때까지 기다렸다가 한 번에 반환해요.

### 왜 invoke()부터 배우나요?

`invoke()`는 가장 단순하고 직관적인 호출 방식이에요. 전화를 걸어서 상대방이 말을 다 끝낼 때까지 기다렸다가 한 번에 듣는 것과 같아요. 반면 `stream()`은 실시간 통역처럼 한 마디씩 즉시 전달받는 방식이에요. 먼저 단순한 `invoke()`를 이해하고 나면 다른 방식들이 자연스럽게 이해돼요.

메시지는 두 가지 방식으로 전달할 수 있어요:
- **튜플 형식**: `("role", "content")` — 간결하고 직관적이에요
- **메시지 객체**: `HumanMessage("...")` — 명시적이고 타입 안전해요

> 🔑 **핵심 개념**: `invoke()`의 응답은 항상 `AIMessage` 객체예요. `response.content`로 텍스트를, `response.usage_metadata`로 토큰 사용량을 확인할 수 있어요.


In [5]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: invoke(): 동기 방식 전체 응답 받기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
messages = [
    ("system", "You are a helpful assistant. Answer in Korean."),
    ("human", "대한민국의 수도는 어디야?"),
]

res = model.invoke(messages)
res

AIMessage(content='대한민국의 수도는 서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 29, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1b483c1ec4', 'id': 'chatcmpl-DeYeOosfxHCAQUNczxwjd5e41GgVO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1a5b-dc00-7671-8043-86619c4a2343-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: AIMessage 응답 구조 살펴보기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 스트리밍: stream()

`stream()`은 토큰이 생성되는 즉시 출력해요. 긴 응답을 기다리지 않아도 되므로 사용자 경험이 크게 향상돼요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: stream(): 실시간 스트리밍 출력
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
for chunk in model.stream(messages):
    print(chunk.content, end="")

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 스트리밍 헬퍼 함수 직접 구현
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 비동기 처리: ainvoke()와 astream()

Python의 `async/await` 구문으로 비동기 처리를 해요. 여러 모델 호출을 동시에 처리하거나 I/O 바운드 작업과 병렬로 실행할 때 유용해요.

> 🔑 **핵심 개념**: Jupyter 노트북은 기본적으로 이벤트 루프가 실행 중이에요. 그래서 `await`를 최상위 수준에서 바로 사용할 수 있어요. Python 스크립트에서는 `asyncio.run()`으로 감싸야 해요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ainvoke(): 비동기 동기 호출
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: astream(): 비동기 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 배치 처리: batch()

`batch()`는 여러 요청을 한 번에 병렬로 처리해요. 다량의 데이터를 처리할 때 순차 호출보다 훨씬 효율적이에요.

### batch() vs batch_as_completed() 비교

| 구분 | `batch()` | `batch_as_completed()` |
|------|-----------|----------------------|
| 반환 순서 | 입력 순서와 동일 | 완료된 순서대로 |
| 반환 형태 | `list[AIMessage]` | `Iterator[(index, AIMessage)]` |
| 사용 시점 | 순서가 중요할 때 | 빠른 응답부터 처리할 때 |
| 속도 | 가장 느린 요청 기준 | 완료 즉시 반환 |


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: batch(): 여러 요청을 병렬로 처리
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: batch_as_completed(): 완료 순서대로 결과 반환
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. Tool Calling — 도구 호출

### 왜 Tool Calling이 필요한가요?

LLM은 학습 데이터에 기반해 답변하므로 **실시간 정보**(오늘 날씨, 최신 주가)를 알 수 없고, **정확한 계산**(복잡한 수식)도 보장하지 못해요. Tool Calling은 이 한계를 극복하기 위해 LLM이 "나 혼자 답하기 어려우니 외부 도구를 사용하겠다"고 판단하는 메커니즘이에요. 마치 의사가 진단을 내리기 위해 혈액 검사를 의뢰하는 것처럼, LLM도 필요한 정보를 외부에 요청할 수 있어요.

Tool Calling은 LLM이 외부 함수를 호출할 수 있게 해주는 기능이에요. 모델은 사용자 질문을 보고 어떤 도구를 호출할지, 어떤 인자를 넘길지 스스로 결정해요.


In [ ]:
from pydantic import BaseModel, Field

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Tool Calling: Pydantic 모델로 도구 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
class GetWeather(BaseModel):
    """입력받은 지역의 현재 날씨를 가져와"""

    location: str = Field(
        ...,
        description="도시와 국가명. 예: Seoul, South Korea"
    )

class SearchWeb(BaseModel):
    """쿼리를 검색해라."""

    query: str = Field(
        ...,
        description="검색할 질의"
    )
    num_results: int = Field(
        default=5,
        description="반환할 최대 검색 결과 수"
    )

tools = [GetWeather, SearchWeb]
model_with_tools = model.bind_tools(tools)

In [12]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Tool Call 응답 구조 이해하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.
res = model_with_tools.invoke("서울의 현재 날씨는?")

res.content
print(f' ==> [Line 6]: \033[38;2;185;94;121m[res.content]\033[0m({type(res.content).__name__}) = \033[38;2;43;103;226m{res.content}\033[0m')
res.tool_calls
print(f' ==> [Line 8]: \033[38;2;91;133;233m[res.tool_calls]\033[0m({type(res.tool_calls).__name__}) = \033[38;2;27;146;184m{res.tool_calls}\033[0m')

# ---------------------------------------------------
# Tool Call 응답 구조 이해하기
# ---------------------------------------------------
# 도구 호출이 여러 개일 수도 있어요 (병렬 도구 호출)
if res.tool_calls:
    for tc in res.tool_calls:
        print(f"도구 이름: {tc['name']}")
        print(f"인자(args): {tc['args']}")
        print(f"호출 ID: {tc['id']}")
        # 이 id는 ToolMessage로 응답할 때 필요해요!
        print(f"타입: {tc['type']}")
        print()


 ==> [Line 6]: [res.content](str) = 
 ==> [Line 8]: [res.tool_calls](list) = [{'name': 'GetWeather', 'args': {'location': 'Seoul, South Korea'}, 'id': 'call_5Idc60HXLpcRUMI9liLx8OC0', 'type': 'tool_call'}]
도구 이름: GetWeather
인자(args): {'location': 'Seoul, South Korea'}
호출 ID: call_5Idc60HXLpcRUMI9liLx8OC0
타입: tool_call



In [13]:
# ============================================================
# TODO: 계산기 도구를 만들어서 모델에 바인딩해보세요
# 힌트: Pydantic BaseModel로 Calculate 클래스를 정의하고
#       expression 필드(str)를 추가해보세요
# 예상 결과: "15 + 27을 계산해줘" 질문에 tool_calls가 생성되어야 해요
# ============================================================

# ============================================================
# 구현 예시: 계산기 도구를 만들어서 모델에 바인딩
# ============================================================
from pydantic import BaseModel, Field


class Calculate(BaseModel):
    """수학식을 계산해야 할 때 사용하세요."""

    expression: str = Field(description="계산할 수학식. 예: '15 + 27'")


model_calc = model.bind_tools([Calculate])
response_calc = model_calc.invoke("15 + 27을 계산해줘")

print("도구 호출 수:", len(response_calc.tool_calls))
for call in response_calc.tool_calls:
    print(call)

# TODO: 모델에 바인딩하고 테스트해보세요
model_calc = model.bind_tools([Calculate])
response_calc = model_calc.invoke("15 + 27을 계산해줘")
print(response_calc.tool_calls)


도구 호출 수: 1
{'name': 'Calculate', 'args': {'expression': '15 + 27'}, 'id': 'call_zyuymU75VG0gm6kDoTkXqunM', 'type': 'tool_call'}
[{'name': 'Calculate', 'args': {'expression': '15 + 27'}, 'id': 'call_itUKWNMqskpShOzfElBsddEC', 'type': 'tool_call'}]


## 7. 구조화 출력: with_structured_output()

`with_structured_output()`은 모델의 응답을 미리 정의한 Pydantic 스키마에 맞게 파싱해요. 정보 추출, 데이터 파싱, API 응답 생성에 매우 유용해요.


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: with_structured_output(): 구조화된 응답 받기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

# SentimentAnalaysis 출력 파서 만들어 보기
#  - 감성 분류 : positive, negative, neutral 중 하나
#  - 분류 신뢰도 0.0 ~ 1.0
#  - 핵심 감성 표현 목록 (최대 3개)
#  - 감성 분석 요약 ( 한 문장 )

review = "이 제품은 정말 대단해요! 배송도 빠르고 품질도 최고예요. 완전 만족합니다."

## 8. 토큰 사용량 추적

대규모 애플리케이션에서는 비용 관리를 위해 토큰 사용량을 추적해야 해요. `UsageMetadataCallbackHandler`는 여러 모델 호출에 걸쳐 누적 사용량을 모델별로 자동 집계해요.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import UsageMetadataCallbackHandler

model_gpt4o_mini = init_chat_model("openai:gpt-4o-mini")    # 소형 모델
model_gpt4o = init_chat_model("openai:gpt-4o")    # 대형 모델 (주석 처리: 비용 절감)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: UsageMetadataCallbackHandler: 토큰 사용량 추적
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.

res

# 누적 토큰 사용량 (모델별):
for model_name, usage in callback.usage_metadata.items():
    print(f"\n모델: {model_name}")
    print(f"  입력 토큰: {usage['input_tokens']}")
    print(f"  출력 토큰: {usage['output_tokens']}")
    print(f"  합계: {usage['total_tokens']}")

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: get_usage_metadata_callback: 컨텍스트 매니저 방식
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. config 파라미터 — 런타임 설정

`config` 파라미터를 통해 런타임에 모델 호출 동작을 세밀하게 제어할 수 있어요. LangSmith에서 실행을 추적하고 필터링하는 데 특히 유용해요.

| `config` 키 | 설명 |
|---|---|
| `run_name` | LangSmith 실행 이름 |
| `tags` | 분류·필터링용 태그 목록 |
| `metadata` | 임의의 키-값 메타데이터 (사용자 ID, 세션 등) |
| `callbacks` | 실행 중 호출될 콜백 핸들러 목록 |


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: config 파라미터 사용
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 10. 멀티모달 입력 — 이미지 처리

최신 LLM은 텍스트뿐 아니라 이미지도 처리할 수 있어요. LangChain V1에서는 `HumanMessage`의 `content`에 이미지 데이터를 포함시켜서 멀티모달 요청을 만들어요.

> 💡 **팁**: 외부 래퍼 클래스에 의존하지 않고 표준 LangChain V1 방식으로 직접 구현하는 것이 좋아요. 내부 동작을 이해할 수 있고 외부 의존성도 줄일 수 있어요.


In [ ]:
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

vision_model = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 멀티모달: URL 이미지 분석
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
    from langchain.chat_models import init_chat_model
    from langchain.messages import HumanMessage, SystemMessage

    llm = init_chat_model(model_name)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 멀티모달 헬퍼 함수 직접 구현
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 11. Logprobs — 토큰 확률 분포

`logprobs` 옵션을 활성화하면 모델이 각 토큰을 생성할 때의 로그 확률을 받을 수 있어요. 모델의 확신도를 측정하거나 불확실성을 분석할 때 유용해요.

> 🔑 **핵심 개념**: Logprobs는 현재 OpenAI 모델에서만 지원해요. `bind(logprobs=True)`로 활성화하면 응답 메타데이터에 토큰별 로그 확률이 포함돼요. 로그 확률을 확률로 변환하려면 `math.exp(logprob)` 또는 `math.exp(logprob) * 100` (%)로 계산해요.


In [ ]:
# ---------------------------------------------------
# Logprobs: 토큰 확률 분포 확인
# ---------------------------------------------------
# 토큰 확률을 추출하는 함수를 직접 구현해요
import math
from langchain_openai import ChatOpenAI


def extract_token_probabilities(response) -> dict:
    """AIMessage에서 토큰별 확률을 추출하는 함수예요.
    
    logprobs가 활성화된 모델의 응답에서만 동작해요.
    
    Args:
        response: logprobs가 포함된 AIMessage 객체
    
    Returns:
        {'tokens': [...], 'probabilities': [...]} 딕셔너리
    """
    logprobs_data = response.response_metadata.get("logprobs", {})
    if not logprobs_data or "content" not in logprobs_data:
        return {"tokens": [], "probabilities": []}
    
    tokens = []
    probabilities = []
    
    for token_info in logprobs_data["content"]:
        token = token_info["token"]
        logprob = token_info["logprob"]
        prob_percent = round(math.exp(logprob) * 100, 2)  # 퍼센트로 변환
        tokens.append(token)
        probabilities.append(prob_percent)
    
    return {"tokens": tokens, "probabilities": probabilities}


# logprobs=True로 바인딩: OpenAI 모델에서만 지원해요
logprobs_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,  # 확정적인 답변을 원할 때
).bind(logprobs=True)

# 예/아니오 질문으로 모델의 확신도 측정
response_lp = logprobs_model.invoke(
    "대한민국의 수도는 부산입니까? '예' 또는 '아니오'로만 답해주세요."
)

print(f"응답: {response_lp.content}")
print()

# 토큰 확률 추출
prob_data = extract_token_probabilities(response_lp)

if prob_data["tokens"]:
    # 토큰별 확률 분포:
    for token, prob in zip(prob_data["tokens"], prob_data["probabilities"]):
        bar = "█" * int(prob / 5)  # 간단한 막대 그래프
        print(f"  {token!r:15} | {bar} {prob:.1f}%")
else:
    # logprobs 데이터가 없어요.
    pass

## 12. 커스텀/사내 모델 서버 연결 전략

이 과정에서는 별도 로컬 런타임 도구 실습을 다루지 않아요. 커스텀 엔드포인트나 사내 모델 서버가 필요하면 다음 섹션의 **OpenAI 호환 API** 패턴처럼 `base_url`을 지정해 연결합니다.

> 💡 **실무 팁**: 제공자가 OpenAI 호환 Chat Completions API를 제공하면 `ChatOpenAI(base_url=..., api_key=...)` 형태로 같은 LangChain 인터페이스를 유지할 수 있어요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: OpenAI 호환 API로 커스텀/사내 모델 서버 연결
# 힌트: ChatOpenAI의 base_url, api_key, model 인자를 사용해 같은 LangChain 인터페이스로 연결하세요.


## 13. OpenAI 호환 API — 커스텀 엔드포인트

`ChatOpenAI`의 `base_url`을 변경하면 LM Studio, vLLM, OpenRouter 등 OpenAI API 형식을 따르는 서비스에 연결할 수 있어요.

### 파라미터 전달 방식

| 방식 | 용도 | 예시 |
|------|------|------|
| `model_kwargs` | 표준 OpenAI 파라미터 | `max_completion_tokens`, `stream_options` |
| `extra_body` | 제공자 고유 파라미터 | vLLM의 `use_beam_search`, LM Studio의 `ttl` |


In [ ]:
# ---------------------------------------------------
# OpenAI 호환 API: base_url 변경으로 다양한 서비스 연결
# ---------------------------------------------------
from langchain_openai import ChatOpenAI

# LM Studio 연결 예시 (로컬 GUI LLM 서버)
# lm_studio_model = ChatOpenAI(
#     base_url="http://localhost:1234/v1",    # LM Studio 기본 주소
#     api_key="lm-studio",                    # 아무 문자열이나 가능
#     model="mlx-community/Llama-3.2-3B-Instruct",
#     extra_body={"ttl": 300}                 # LM Studio 고유 파라미터
# )

# vLLM 연결 예시 (고성능 추론 서버)
# vllm_model = ChatOpenAI(
#     base_url="http://localhost:8000/v1",    # vLLM 기본 주소
#     api_key="EMPTY",                        # vLLM은 API 키 불필요
#     model="meta-llama/Llama-2-7b-chat-hf",
#     extra_body={                            # vLLM 고유 파라미터
#         "use_beam_search": True,
#         "best_of": 4
#     }
# )

# OpenRouter 연결 예시 (다양한 모델 통합 서비스)
# openrouter_model = ChatOpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key="sk-or-...",                   # OpenRouter API 키
#     model="anthropic/claude-3-haiku",      # 모델명 형식이 달라요
# )

# model_kwargs vs extra_body 구분 예시
# model_kwargs: 표준 OpenAI API 파라미터 (최상위 payload에 병합)
# model_with_kwargs = ChatOpenAI(
#     model="gpt-4o-mini",
#     model_kwargs={
#         "stream_options": {"include_usage": True},
#         "max_completion_tokens": 300,
#     }
# )

# OpenAI 호환 API 설정 예시 코드입니다.
# 주석을 해제하고 실제 서버 주소로 변경하면 바로 사용할 수 있어요.


## 14. reasoning_effort — 추론 강도 조절

OpenAI의 추론 모델(o1, o3 등)은 `reasoning_effort`로 추론 깊이를 조절할 수 있어요. 비용과 품질 사이의 균형을 맞출 수 있어요.

> 🔑 **핵심 개념**: `reasoning_effort`는 추론 모델의 "생각하는 시간"을 조절해요. `high`는 더 깊이 생각해서 정확도가 높지만 느리고 비싸요. 수학, 코딩, 논리적 추론 문제에 적합해요.

| `reasoning_effort` | 설명 | 권장 용도 |
|---|---|---|
| `minimal` | 최소 추론 (가장 빠름) | 단순 질문 |
| `low` | 낮은 추론 | 가벼운 작업 |
| `medium` | 중간 추론 (균형) | 일반적 작업 |
| `high` | 깊은 추론 | 수학·코딩·복잡한 로직 |


In [ ]:
from langchain_openai import ChatOpenAI

reasoning_model = ChatOpenAI(

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: reasoning_effort: 추론 강도 조절
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`init_chat_model`**: `"provider:model"` 형식으로 20+ 제공자를 통일된 방식으로 초기화해요. V1의 가장 중요한 변화예요
- **5가지 호출 방식**: `invoke`(동기), `stream`(실시간), `ainvoke`/`astream`(비동기), `batch`(병렬)을 상황에 맞게 선택해요
- **Tool Calling**: `bind_tools()`로 모델에 도구를 알려주고, `response.tool_calls`로 호출 정보를 받아요
- **구조화 출력**: `with_structured_output(PydanticModel)`으로 일관된 구조의 응답을 보장해요
- **토큰 추적**: `UsageMetadataCallbackHandler`나 `get_usage_metadata_callback`으로 비용을 모델별로 집계해요
- **멀티모달**: `HumanMessage`의 `content`에 이미지 블록을 포함시켜서 이미지를 분석해요
- **Logprobs**: `bind(logprobs=True)`로 토큰 확률 분포를 얻어서 모델의 확신도를 측정해요
- **OpenAI 호환 API**: `base_url` 변경으로 LM Studio, vLLM, OpenRouter 등 다양한 서버에 연결해요


## 다음 노트북 예고

다음 `03-Messages.ipynb`에서는 **메시지 타입과 Content Blocks**를 배워요. `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage` 4가지 메시지 타입의 구조와 멀티모달 Content Blocks를 활용하는 방법을 자세히 다뤄요.
